# Import Cell

In [1]:
import torch
import numpy as np
import time
device='cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# Difference Between GPU and CPU usage

The format specifier :.4f determines how a decimal number (floating-point number) is formatted and displayed in programming languages like Python.
It means the number will be displayed as a fixed-point decimal rounded to exactly 4 decimal places.
Here is how the syntax breaks down:
- : Signals the start of the formatting rules.
- .4 Specifies the precision, forcing exactly 4 digits after the decimal point.
- f Represents "fixed-point" notation, which displays the number as a standard decimal rather than in scientific notation.

We can also use %%time as a header to record the time taken for execution.
What happens realtime is this, cpu executes but there are some waiting times in between blocks of execution. 
```CPU times``` shows cpu calculations only while ```Wall times``` include those wait times as well.

In [2]:
%%time
# we are using time module of python
start_time=time.time() # It will store the time from epoch in ms on execution at this line

# Matrix Operation Here
zeros=torch.zeros(1,1)
end_time=time.time()

delta_time=end_time-start_time
print(f'{delta_time:.4f}')


0.0001
CPU times: user 225 μs, sys: 45 μs, total: 270 μs
Wall time: 214 μs


```0.0000```
- ```CPU times```: user 136 μs, sys: 5 μs, total: 141 μs; Calculates the cpu usage
- ```Wall time```: 134 μs; Calculates the total time from execution and side-effect appearing

## Let's Test which one is faster on gpu, torch or numpy

In [4]:
torch_rand1=torch.rand((10000,10000),dtype=torch.float64).to(device)
# I don't know what to device does. probably assigning it on our global variable 'device' in the import cell
torch_rand2=torch.rand((10000,10000),dtype=torch.float64).to(device)
# Out guy made a mistake, he shoved tensors into np.multiply, apples-to-oranges maths
np_rand1=np.random.rand(10000,10000).astype(np.float64)
# This is np.random composite module method .rand(start=,end=). Here we use astype() incase of detype of torch and use np.float32 object as type
np_rand2=np.random.rand(10000,10000).astype(np.float64)

# Time for cuda block;
start_time=time.time()
rand_gpu=(torch_rand1@torch_rand2)
# The above is the huge multiplication(we don't need to see it), @ is torch specific mul
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.00019813 fucking screaming fast!
# For float64 0.00018287 SAME SPEED!

# Time for numpy block;
start_time=time.time()
#rand=np.multiply(np_rand1,np_rand2)
# The above is element wise multiplication, we need to compare numpy object matrix multiplication using @
rand_cpu=np_rand1@np_rand2
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 4.21621251 screaming slow!
# For float64 10.94090414; double as expected

# Let's time numpy elementwise mul
start_time=time.time()
rand_cpu=np.multiply(np_rand1,np_rand2)
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.09828210 okay, but definitely not doing the job!
# For float64 0.17952561; double as expected
# Shape is simple, so cpu is better at it!

0.01671314
11.39632916
0.21614599


## Let's Test a shape of (100,100,100,100) so 4D, let's see who wins

In [5]:
torch_rand1=torch.rand((100,100,100,100),dtype=torch.float64).to(device)
# I don't know what to device does. probably assigning it on our global variable 'device' in the import cell
torch_rand2=torch.rand((100,100,100,100),dtype=torch.float64).to(device)
# Out guy made a mistake, he shoved tensors into np.multiply, apples-to-oranges maths
np_rand1=np.random.rand(100,100,100,100).astype(np.float64)
# This is np.random composite module method .rand(start=,end=). Here we use astype() incase of detype of torch and use np.float32 object as type
np_rand2=np.random.rand(100,100,100,100).astype(np.float64)

# Time for cuda block;
start_time=time.time()
rand_gpu=(torch_rand1@torch_rand2)
# The above is the huge multiplication(we don't need to see it), @ is torch specific mul
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.00027418 slower than 2D!
# For float64 0.00026464 even Faster!!!!

# Time for numpy block;
start_time=time.time()
#rand=np.multiply(np_rand1,np_rand2)
# The above is element wise multiplication, we need to compare numpy object matrix multiplication using @
rand_cpu=np_rand1@np_rand2
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.16696119 great performance!
# For float64 0.35045409; double as expected

# Let's time numpy elementwise mul
start_time=time.time()
rand_cpu=np.multiply(np_rand1,np_rand2)
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.09236670 okay, but definitely not doing the job!
# For float64 0.17640281; double as expected
# Shape is simple, so cpu is better at it!

0.00041032
0.36043477
0.20490050


# Billions of But Simple Computation
When we will have millions or billions of parameters for our model, we will be doing very simple computations.
Above example demonstrates real time how efficient my rtx 3060 ti 8GB gpu is compared to 10700k.

## Let's Now Work on Some torch functions

In [6]:
# Define a probability tensor
probabilities=torch.tensor([0.1,0.9])
# 10% or .1->0.90% or .9->1; probabilities[0],probabilities[1]=0.1,0.9 
# Each probability points to the index of the of the probability in the tensor and they have to add up to 1
# Draw 5 samples from multinomial distribution
samples=torch.multinomial(probabilities,num_samples=10,replacement=True)
# Multinomial takes in a probability tensor object of 1 by 2 as its first argument and creates a 1D tensor consisting of 0s and 1s.
# Multinomials shows the indexes in probability order or bias, remember that wighted walk by mewstro?
# According to our script the index 0 has 10% chance to exist and index 1 has 90% chance to exist, and torch.multinomial returns a random 1D tensor with the indexes as elements(booleans for something else?).
print(samples)

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


We are going to use this to predict what words gonna come next as biases(or random walks).

In [19]:
# torch.cat() concatenates two tensors like lists
tensor=torch.tensor([1,2,3,4])
out=torch.cat((tensor,torch.tensor([5])),dim=0)
out

tensor([1, 2, 3, 4, 5])

Above torch.cat() is same as the concatenation below.

In [20]:
a=[1,2,3,4]
b=[5]
a+b

[1, 2, 3, 4, 5]

## We Are Going To Actually Use This During Generation
1. It's going to start from zero, we are going to use our probability of distribution to pick the first character.
2. Based on the first character we are going to pick the second one and then ```concatenate``` with the first one.

For example if we have 100 characters here and remaining 100 there, we are going to concatenate them using this method. And in the end we will have all of these integers that we have predicted.

## tril Stands For Triangle Lower(Including The Diagonal)

In [ ]:
# tril is gonna multiply the upper part of the diagonal with zero
out=torch.tril(torch.ones(5,5))
out=torch.tril(torch.ones(5,5))
out

# The Triangle Shows Time
First we predict 1 and remaining 4 is zero in the first row but slowly the ones talk with the next one and we have 3 zeros and in the last tow we just finished predicting all 5 of them.
```
[[1., 0., 0., 0., 0.],
 [1., 1., 0., 0., 0.],
 [1., 1., 1., 0., 0.],
 [1., 1., 1., 1., 0.],
 [1., 1., 1., 1., 1.]]
```
In the end we have more and more history to look back into rather than future.
## Premise:
Basically the premise of this is to make sure that we can't communicate with the answere, just like in the exam hall. Opposit case for triu;
```
[[1., 1., 1., 1., 1.],
 [0., 1., 1., 1., 1.],
 [0., 0., 1., 1., 1.],
 [0., 0., 0., 1., 1.],
 [0., 0., 0., 0., 1.]]
```
## triu Stands For Triangle Upper(Including The Diagonal)

In [ ]:
# triu is gonna multiply the lower part of the diagonal with zero
out=torch.triu(torch.ones(5,5))
out

# Mask
When we store a boolean condition with a given tensor containing 1s or 0s, we get a mask, a tensor with trues where boolean satisfies and falses where it doesn't. a mask as such, a tensor of boolean value is insertable into the ```masked_fill()``` method of another tensor object;
```
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
```
The Trues get multiplied element-wise with the 2nd args provided as an float() or int() object and gives a tensor as;
```
tensor([[False,  arg2,   arg2,  arg2,  arg2 ],
        [False,  False,  arg2,  arg2,  arg2 ],
        [False,  False,  False, arg2,  arg2 ],
        [False,  False,  False, False, arg2 ],
        [False,  False,  False, False, False]])
```

In [40]:
mask=torch.tril(torch.ones(5,5))==0
masked=torch.zeros(5,5).masked_fill(mask,float('-inf'))
masked

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

# Exponentiate Refers to Natural Exoinential Function
Since we were just working with `masked_fill` and `float('-inf')`, exponentiation is the very next step in the **Softmax** formula where $e^{0}=0$;

* **Positive numbers grow fast:** Larger scores become much larger weights.
* **Negative numbers become tiny positive fractions:** For example, $e^{-2} \approx 0.1353$.
* **Negative infinity (`-inf`) becomes `0`:** Because $e^{-\infty} = 0$, filling masked spots with `-inf` ensures they turn into `0.0` after exponentiation and get completely ignored by the model.

In [41]:
# Let's use torch.exp().
torch.exp(masked)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

## '-inf' Becomes 0 and 0 becomes 1
```
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])
```
## Transpose is More Like Swapping

In [46]:
input=torch.zeros(2,3,4)
# By default torch.zeros() takes dimension as args.
# We can use method transpose() to swap the dimension value from the input tensor.
# .transpose() takes is index th number of the dimensions and swaps them.
out=input.transpose(0,2)
# Here we just swaped 0th dimension 2 with 2th dimention 4.
input,out

(tensor([[[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]],
 
         [[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]]]),
 tensor([[[0., 0.],
          [0., 0.],
          [0., 0.]],
 
         [[0., 0.],
          [0., 0.],
          [0., 0.]],
 
         [[0., 0.],
          [0., 0.],
          [0., 0.]],
 
         [[0., 0.],
          [0., 0.],
          [0., 0.]]]))

# Stack Tensors on Next Dimension
torch.stack([torch.tensor]) takes in tensor objects and house them serially as rows in the direction of the column space. It can take in tensors as opposed to torch.tensor([list(]) which takes in list objects only.

## Relevance to Bach Size

In [50]:
tensor1=torch.tensor([1,2,3])
tensor2=torch.tensor([4,5,6])
tensor3=torch.tensor([7,8,9])

# Stack along next dimension
stacked=torch.stack([tensor1,tensor2,tensor3])
stacked

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])